## ProblemBuilder Class

LP programming problem building class:

In [1]:
class TropicalLPBuilder:
    """
    LP programming problem builder for tropical algebra
    """
    
    def __init__(self, semiring='minplus', numeric_type='float'):
        """
        Args:
            semiring: 'minplus' o 'maxplus'
            numeric_type: 'float', 'int', 'big_int', 'big_rational'
        """
        self.variables = []
        self.semiring = semiring
        self.numeric_type = numeric_type
        self.objective = None
        self.objective_type = None  # 'minimize' or 'maximize'
        self.constraints = []
        self.basic_point = []
        
    def add_variables(self, *var_names):
        """Add variables to the problem"""
        for var in var_names:
            if var not in self.variables:
                self.variables.append(var)
        print(f"Variables added: {list(var_names)}")
        print(f"Total variables: {self.variables}")
        return self
        
    def minimize(self, expression):
        """Define objective function for minimization"""
        self.objective = expression
        self.objective_type = 'minimize'
        print(f"Objective: minimize {expression}")
        return self
        
    def maximize(self, expression):
        """Define objective function for maximization"""
        self.objective = expression
        self.objective_type = 'maximize'
        print(f"Objective: maximize {expression}")
        return self
        
    def add_constraint(self, left_expr, operator, right_expr):
        """
        Adds a constraint
        
        Args:
            left_expr: left-hand side expression
            operator: '<=' or '>='
            right_expr: right-hand side expression
        """
        constraint = f"{left_expr} {operator} {right_expr}"
        self.constraints.append(constraint)
        print(f"Constraint added: {constraint}")
        return self
        
    def set_basic_point(self, *values):
        """Defines the initial basic point"""
        self.basic_point = list(values)
        print(f"Basic point: {self.basic_point}")
        return self
        
    def to_lp_format(self):
        """Converts the problem to .lp format"""
        lines = []
        
        # Header
        lines.append(f"vars: {', '.join(self.variables)}")
        lines.append(f"semiring: {self.semiring}")
        lines.append(f"numeric: {self.numeric_type}")
        lines.append("lp:")
        
        # Objective
        if self.objective:
            lines.append(f"{self.objective_type} {self.objective};")
        
        # Constraints
        for constraint in self.constraints:
            lines.append(f"{constraint};")
            
        # Basic point
        if self.basic_point:
            values_str = ', '.join(map(str, self.basic_point))
            lines.append(f"basic point = {values_str};")
            
        return '\n'.join(lines)
        
    def save(self, filename):
        """Saves the problem to a .lp file"""
        content = self.to_lp_format()
        with open(filename, 'w') as f:
            f.write(content)
        print(f"Problem saved to: {filename}")
        return self
        
    def display(self):
        """Displays the problem in .lp format"""
        print("=== LP TROPICAL PROBLEM ===")
        print(self.to_lp_format())
        print("===============================")
        return self

## Example 1

Problem with two variables

In [2]:
problem1 = (TropicalLPBuilder(semiring='minplus')
    .add_variables('x', 'y')
    .minimize('x + 3')
    .add_constraint('x + 2', '<=', 'y + 4')
    .set_basic_point(1.0, 2.0)
    .display()
    .save('example_1.lp')
)

Variables added: ['x', 'y']
Total variables: ['x', 'y']
Objective: minimize x + 3
Constraint added: x + 2 <= y + 4
Basic point: [1.0, 2.0]
=== LP TROPICAL PROBLEM ===
vars: x, y
semiring: minplus
numeric: float
lp:
minimize x + 3;
x + 2 <= y + 4;
basic point = 1.0, 2.0;
Problem saved to: example_1.lp


## Example 2

Problem with more variables and contraints

In [ ]:
problem2 = (TropicalLPBuilder(semiring='maxplus')
    .add_variables('x', 'y', 'z')
    .maximize('x + 5')
    .add_constraint('x + 1', '>=', 'y + 2')
    .add_constraint('y + 3', '<=', 'z + 1')
    .add_constraint('x + z', '>=', '6')
    .set_basic_point(0.0, 1.0, 2.0)
    .display()
    .save('example_2.lp')
)

Variables added: ['x', 'y', 'z']
Total variables: ['x', 'y', 'z']
Objective: maximize x + 5
Constraint added: x + 1 >= y + 2
Constraint added: y + 3 <= z + 1
Constraint added: x + z >= 6
Basic point: [0.0, 1.0, 2.0]
=== LP TROPICAL PROBLEM ===
vars: x, y, z
semiring: maxplus
numeric: float
lp:
maximize x + 5;
x + 1 >= y + 2;
y + 3 <= z + 1;
x + z >= 6;
basic point = 0.0, 1.0, 2.0;
Problem saved to: example_2.lp


## Custom Problem Creation



In [3]:
def create_interactive_problem():
    """Create a custom LP problem interactively"""

    print("=== CREATE TROPICAL LP PROBLEM ===\n")

    # Semiring
    semiring = input("Semiring (minplus/maxplus) (default value without text: minplus): ").strip() or 'minplus'

    # Semiring check
    if semiring.lower() not in ['minplus', 'maxplus']:
        print("Invalid semiring. Please choose 'minplus' or 'maxplus'.")
        return

    builder = TropicalLPBuilder(semiring=semiring)

    # Variables
    vars_input = input("Variables (comma separated, e.g.: x,y,z): ").strip()
    variables = [v.strip() for v in vars_input.split(',') if v.strip()]
    

    builder.add_variables(*variables)
    

    # Objective
    obj_type = input("Objective (minimize/maximize): ").strip()

    # Objective type check
    if obj_type.lower() not in ['minimize', 'maximize']:
        print("Invalid objective type. Please choose 'minimize' or 'maximize'.")
        return
    
    obj_expr = input(f"Expression to {obj_type} (e.g.: x + 3): ").strip()

    if obj_type.lower() == 'minimize':
        builder.minimize(obj_expr)
    else:
        builder.maximize(obj_expr)


    # Constraints
    print("\n === ADD CONSTRAINTS ===")

    while True:
        constraint = input("Enter constraints one at a time (format: left_expr <= right_expr. Press ENTER without text to finish): ").strip()
        if not constraint:
            break
            
        # Parse constraint
        if '<=' in constraint:
            left, right = constraint.split('<=')
            builder.add_constraint(left.strip(), '<=', right.strip())
        elif '>=' in constraint:
            left, right = constraint.split('>=')
            builder.add_constraint(left.strip(), '>=', right.strip())
        else:
            print("Invalid constraint format. Use '<=' or '>='.")
    
    # Basic point
    basic_input = input(f" \n Basic point: ({len(variables)} values separated by commas (optional)): ").strip()
    if basic_input:
        try:
            values = [float(v.strip()) for v in basic_input.split(',')]
            builder.set_basic_point(*values)
        except:
            print("Invalid values for the basic point")

    # Save
    filename_input = input("\n File name (without extension): ").strip()
    
    if not filename_input:
        print("Invalid file name.")
        return
    else:
        filename = filename_input + '.lp'
        builder.display().save(filename)


    return builder


In [4]:
my_problem = create_interactive_problem()

=== CREATE TROPICAL LP PROBLEM ===

Invalid semiring. Please choose 'minplus' or 'maxplus'.
